# 統計学特講

## 03 GDPと成長率

### 目的
- 支出側GDPを計算する
- GDPの三面等価を確認する
- GDP成長率を計算する

### 注意点
- ここでのGDPは全て名目値である

### 前準備

In [ ]:
import pandas as pd              # データフレーム関連のおまじない
import numpy as np               # 数値計算関連のおまじない
import matplotlib.pyplot as plt  # グラフ描画関連のおまじない

### 3-1 GDPの復習

#### GDPとは

GDP（Gross Domestic Product，国内総生産）は，一定期間内に国内で新しく生み出された付加価値の合計である．

同じ経済活動を別の側面から見ているため，GDPには次の3つの見方がある．

- 生産面：どれだけ生産されたか
- 支出面：生産されたものがどのように使われたか
- 分配面：生産で得られた所得がどのように分配されたか

この3つは理論上同じ値になる．これを**三面等価**という．

#### 支出側GDP

支出側から見ると，GDPは次のように表される．

$$\text{GDP} = C + I + G + X - M$$

- $C$：民間消費 (**c**onsumption)
- $I$：投資 (**i**nvestment)
- $G$：政府支出 (**g**overnment spending)
- $X$：輸出 (e**x**ports)
- $M$：輸入 (i**m**ports)

輸入は海外で生産されたものなので，国内で生み出された付加価値を測るGDPからは差し引く．

### 3-2 データを読む

#### CSVデータをデータフレームに読み込み

In [ ]:
data_url = "https://raw.githubusercontent.com/tnarumi/lecture-statistics/main/data/gdp_expenditure_components_japan.csv"
df_gdp_components = pd.read_csv(data_url)

#### データの確認

##### 先頭の確認

数字の単位は **兆円**

In [ ]:
df_gdp_components.head()

##### 末尾の確認

In [ ]:
df_gdp_components.tail()

##### 特定の年度だけ確認

イコール記号が **2つ** であることに注意

In [ ]:
df_gdp_components[df_gdp_components["year"] == 2020]

##### データフレームの概要を確認

In [ ]:
df_gdp_components.info()

##### 列名の確認

In [ ]:
df_gdp_components.columns

このデータでは，次の列を使う．

- year：年
- consumption：民間消費
- investment：投資
- government_spending：政府支出
- exports：輸出
- imports：輸入

##### 基本統計量の一覧表示

In [ ]:
df_gdp_components.describe()

### 3-3 支出側GDPを計算する

#### 純輸出の計算

純輸出 (net exports) は，輸出 (exports) から輸入 (imports) を引いた値である．

In [ ]:
df_gdp_components["net_exports"] = df_gdp_components["exports"] - df_gdp_components["imports"]
df_gdp_components.head()
# df_gdp_components[["year", "exports", "imports", "net_exports"]].head() #一部だけ表示したいときは、列をリストで指定する

#### 支出側GDP (GDP (expenditure based)) の計算

$$\text{GDP} = C + I + G + (X - M)$$

In [ ]:
df_gdp_components["gdp_expenditure"] = (
    df_gdp_components["consumption"]
    + df_gdp_components["investment"]
    + df_gdp_components["government_spending"]
    + df_gdp_components["net_exports"]
)

df_gdp_components.head()
#df_gdp_components[["year", "consumption", "investment", "government_spending", "net_exports", "gdp_expenditure"]].head()

#### 生産側GDPや分配側GDPとの関係

三面等価により，理論上は支出側GDPと生産側GDPは同じ値になる．ただし，実際の公表統計では，集計方法や統計上の不突合により，完全には一致しない．

公表されているGDPをウェブサイト（例えば，
[GDPの年次推計に関するダッシュボード](https://www.digital.go.jp/resources/japandashboard/gdp-annual-estimates)）で参照し，今回計算した値と同程度であることを確認せよ．

### 3-4 成長率を計算する

#### GDP成長率

GDP成長率 $g$ は，前年と比べてGDPが何％増減したかを表す．

$$g = \frac{\text{今年のGDP} - \text{前年のGDP}}{\text{前年のGDP}} \times 100$$

#### 1年分を手計算で確認

2025年の成長率を，2024年と2025年のGDPから計算してみる．

In [ ]:
gdp_2024 = 634.7511
gdp_2025 = 663.4353

(gdp_2025 - gdp_2024) / gdp_2024 * 100

データフレームとして一括計算

In [ ]:
# 前年のGDPを新しい列として作る
df_gdp_components["gdp_previous_year"] = df_gdp_components["gdp_expenditure"].shift(1)
df_gdp_components[["year", "gdp_expenditure", "gdp_previous_year"]].head()

* 最初の年は前年のデータがないため，前年GDPはNaNになる．NaNはデータが存在しないことを表す．

In [ ]:
# 定義式に沿って成長率を計算する
df_gdp_components["gdp_growth_rate"] = (
    (df_gdp_components["gdp_expenditure"] - df_gdp_components["gdp_previous_year"])
    / df_gdp_components["gdp_previous_year"]
    * 100
)
df_gdp_components[["year", "gdp_expenditure", "gdp_growth_rate"]].head(10)

* 前年GDPがNaNのところは，そこから計算したGDPもNaNとなる．

#### pct_changeを使った計算

`pct_change()` は，1つ前の行からの変化率を計算する関数である．定義式で計算した結果と同じになることを確認する．

In [ ]:
df_gdp_components["gdp_growth_rate_pct_change"] = df_gdp_components["gdp_expenditure"].pct_change() * 100
df_gdp_components[["year", "gdp_expenditure", "gdp_growth_rate", "gdp_growth_rate_pct_change"]].head(10)

#### 直近7年分の成長率を確認

In [ ]:
df_gdp_components[["year", "gdp_expenditure", "gdp_growth_rate"]].tail(7)

#### 成長率が高い年・低い年を確認

In [ ]:
df_gdp_components.nlargest(5, "gdp_growth_rate")[["year", "gdp_growth_rate"]]

In [ ]:
df_gdp_components.nsmallest(5, "gdp_growth_rate")[["year", "gdp_growth_rate"]]

主な出来事一覧

| 年 | 出来事 |
|---|---|
| 1998 | アジア通貨危機，日本の金融不安，不良債権問題 |
| 1999 | 金融システム不安後の景気低迷，デフレ圧力 |
| 2001 | ITバブル崩壊後の世界的な景気減速 |
| 2002 | デフレ，不良債権問題，国内景気の停滞 |
| 2008 | リーマンショック，世界金融危機の発生 |
| 2009 | リーマンショック後の世界同時不況 |
| 2010 | 世界金融危機後の景気回復局面 |
| 2011 | 東日本大震災，福島第一原発事故，サプライチェーン寸断 |
| 2014 | 消費税率引き上げ（5%から8%） |
| 2015 | アベノミクス期，円安，消費税率引き上げ後の持ち直し |
| 2020 | 新型コロナウイルス感染症の拡大，緊急事態宣言 |
| 2021 | コロナ禍からの経済活動再開，前年からの反動 |
| 2023 | 新型コロナの5類移行，インバウンド回復，物価上昇 |
| 2024 | 日銀のマイナス金利政策解除，円安，物価上昇 |
| 2025 | 賃上げ，物価上昇，設備投資・消費動向への注目 |

### 3-5 可視化

#### GDPの時系列プロット

In [ ]:
df_gdp_components.plot(x="year", y="gdp_expenditure", figsize=(10, 4), marker="o", grid=True)

plt.xlabel("year")
plt.ylabel("GDP")
plt.title("GDP calculated from expenditure components")
plt.show()

#### GDP成長率の時系列プロット

In [ ]:
df_gdp_components.plot(x="year", y="gdp_growth_rate", figsize=(10, 4), marker="o", grid=True)
# df_gdp_components.plot(x="year", y="gdp_growth_rate", figsize=(10, 4), kind="bar", grid=True)

plt.axhline(0, color="black", linewidth=1)
plt.xlabel("year")
plt.ylabel("growth rate (%)")
plt.title("GDP growth rate")
plt.show()

#### 支出項目の時系列プロット

In [ ]:
df_gdp_components.plot(
    x="year",
    y=["consumption", "investment", "government_spending", "exports", "imports"],
    figsize=(10, 5),
    marker="o",
    grid=True
)

plt.xlabel("year")
plt.ylabel("value")
plt.ylim(0, 450)
plt.yticks(range(0, 451, 100))
plt.title("Expenditure components")
plt.legend(loc="best")
plt.show()

### 3-6 練習用

課題：下記内容を確認しましょう．
1. 2020年のGDPはいくらか
2. 2020年のGDP成長率を，2019年のGDPと2020年のGDPから計算せよ
3. 純輸出が最も大きい年を確認せよ
4. 消費とGDPを同じグラフに描け
5. GDP成長率がマイナスの年には何が起きているか考えよ

In [ ]:
# ここにコードを各自で追加してください
